In [ ]:
from typing import Any, Literal, TypeVar

In [ ]:
import numpy as np
from bloqade.analysis.fidelity import FidelityAnalysis
from bloqade.types import Qubit
from kirin.dialects import ilist
from kirin.dialects.ilist import IList

In [ ]:
from bloqade import squin
from bloqade.gemini.common.dialects import qubit
from bloqade.lanes.arch.gemini.logical import steane7_initialize
from bloqade.lanes.arch.gemini.physical import get_arch_spec
from bloqade.lanes.noise_model import generate_simple_noise_model
from bloqade.lanes.passes import ASAPPlacePass
from bloqade.lanes.pipeline import PhysicalPipeline
from bloqade.lanes.transform import MoveToSquinPhysical
from bloqade.lanes.visualize import debugger

In [ ]:
kernel = squin.kernel.add(qubit)
kernel.run_pass = squin.kernel.run_pass

In [ ]:
LogicalQubit = ilist.IList[Qubit, Literal[7]]

# how did we not implement this??? -- because it's not free -- state prep circ doesn't really align well with hypercube
# you could change the layout but why not change the kernel
@squin.kernel
def steane7_initialize_custom(
    theta: float, phi: float, lam: float, qubits: ilist.IList[Qubit, Literal[7]]
):
    # debug.info("Begin Steane7 Initialize")
    squin.u3(theta, phi, lam, qubits[6])
    squin.broadcast.sqrt_y_adj(qubits[:6])
    evens = qubits[::2]  # [0, 2, 4, 6]
    odds = qubits[1::2]  # [1, 3, 5]

    # Fixed: CZ pairs should be (1,2), (3,4), (5,6) not (1,0), (3,2), (5,4)
    squin.broadcast.cz(ilist.IList([qubits[1], qubits[2], qubits[5]]), ilist.IList([qubits[3], qubits[4], qubits[6]]))
    squin.sqrt_y(qubits[6])
    squin.broadcast.cz(ilist.IList([qubits[0], qubits[3], qubits[4]]), ilist.IList([qubits[2], qubits[5], qubits[6]]))
    squin.broadcast.sqrt_y(qubits[2:])
    squin.broadcast.cz(evens[:-1], odds)
    squin.broadcast.sqrt_y(ilist.IList([qubits[1], qubits[3], qubits[4]]))
    # NOTE: observable frame changes slightly AND you have to keep track of this change in the detectors/observables computation (!!)
    squin.x(qubits[2])
    squin.broadcast.z(ilist.IList([qubits[1], qubits[5]]))
    # debug.info("End Steane7 Initialize")

In [ ]:
def steane_slot_allocator():
    """Generates a qubit allocator for logical qubits.
    Tries to allocate logical qubits into the architecture in an efficient way to
    make parallelism in the logical gagdets as easily as possible in the move compiler.

    """
    # canonical slot order
    slot_words = ilist.IList([0, 4, 8, 12, 16, 2, 6, 10, 14, 18])

    slots = IList(
        [
            IList([(0, word_id, site_id) for site_id in range(7)])
            for word_id in slot_words
        ]
    )

    @kernel
    def qalloc_slot(
        slot_index: int, theta: float, phi: float, lam: float
    ) -> LogicalQubit:
        def allocate_at(address: tuple[int, int, int]):
            return qubit.new_at(address[0], address[1], address[2])

        addresses = slots[slot_index]

        reg = ilist.map(allocate_at, addresses)

        steane7_initialize(theta, phi, lam, reg)

        return reg

    @kernel
    def qalloc(
        slot_indices: list[int] | IList[int, Any],
        theta: float = 0.0,
        phi: float = 0.0,
        lam: float = 0.0,
    ) -> IList[LogicalQubit, Any]:

        def _inner(slot_index: int):
            return qalloc_slot(slot_index, theta, phi, lam)

        return ilist.map(_inner, slot_indices)

    return qalloc, qalloc_slot

In [ ]:
qalloc, qalloc_slot = steane_slot_allocator()

In [ ]:
N = TypeVar("N")

In [ ]:
@kernel
def flat(
    reg: ilist.IList[LogicalQubit, Any],
) -> ilist.IList[Qubit, Any]:
    """Flatten a logical register into a single list of physical qubits"""

    def _inner(cumulant, ele):
        return cumulant + ele

    return ilist.foldl(_inner, reg, ilist.IList([]))

In [ ]:
@kernel
def cx(controls: ilist.IList[LogicalQubit, N], targets: ilist.IList[LogicalQubit, N]):
    """Efficient broadcasted cx gate over steane logical qubits"""
    squin.broadcast.cx(flat(controls), flat(targets))

In [ ]:
# NOTE: I don't really get the point of this. If you always broadcast across your logical qubits then what's the point.
# this is really for the use case where you don't want to broadcast across your logical qubits.. which, I can't think of anything at the moment
# for this use case.
# ^ it seems like the point of programming at the physical level is that we can test and play with our physical compiler
@kernel
def cz(controls: ilist.IList[LogicalQubit, N], targets: ilist.IList[LogicalQubit, N]):
    """Efficient broadcasted cz gate over steane logical qubits"""
    squin.broadcast.cz(flat(controls), flat(targets))

In [ ]:
@kernel
def measure_logical_reg(logical_reg: ilist.IList[LogicalQubit, Any]):
    """Helper function to get around single measurement restriction of kernel.
    first flatten the logical register into physical qubits, then reconstruct
    the groups of physical measurements into groups related to logical qubits.

    """
    # measurements must be flattened, only one measurement is allowed!
    measurements = squin.broadcast.measure(flat(logical_reg))
    logical_groups = []
    for i in range(len(logical_reg)):
        logical_groups = logical_groups + [measurements[7 * i : 7 * i + 7]]

    return logical_groups

In [ ]:
@kernel(typeinfer=True)
def main():
    reg = qalloc([0, 1, 2, 3, 4], 0.0, 0.0, 0.0)

    # squin.broadcast.h(reg[0])
    # cx(reg[:1], reg[1:2])
    # cx(reg[:2], reg[2:])
    # from the MSD paper
    # NOTE: if programming on the physical level, you ALWAYS have to broadcast your single qubit gates, WITH flattening.

    squin.broadcast.sqrt_x(flat(ilist.IList([reg[0], reg[1], reg[4]])))
    cz(ilist.IList([reg[0], reg[2]]), ilist.IList([reg[1], reg[3]]))
    squin.broadcast.sqrt_y(flat(ilist.IList([reg[1], reg[3]])))
    cz(ilist.IList([reg[1], reg[3]]), ilist.IList([reg[2], reg[4]]))
    squin.broadcast.sqrt_x_adj(flat(ilist.IList([reg[1]])))
    cz(ilist.IList([reg[0], reg[3]]), ilist.IList([reg[1], reg[4]]))
    squin.broadcast.sqrt_x_adj(flat(reg))
    return measure_logical_reg(reg)

In [ ]:
move_mt = PhysicalPipeline(place_opt_type=ASAPPlacePass).emit(main)

In [ ]:
debugger(
    move_mt,
    arch_spec=get_arch_spec(),
)

In [ ]:
noise_model = MoveToSquinPhysical(
    get_arch_spec(),
    noise_model=generate_simple_noise_model(loss=False),
    aggressive_unroll=True,
).emit(move_mt)

In [ ]:
fid = FidelityAnalysis(noise_model.dialects)
fid.run(noise_model)
# note frange.min/max is only used when there is control flow
print(
    "Fidelity max: ",
    np.exp(sum(np.log(frange.max) for frange in fid.gate_fidelities)),
)